#### Gradiente reconstruido desde las PCs de presión: ¿pierde el XGB? ¿y arregla el generador?
Motivación (test por grupos, 20-nov): con un generador lineal (surrogado o VAR), el máx. anual de f se conserva si el XGB usa **solo msl** (ratio 0.98), pero cae si usa **gradiente** (0.78) o las 30 (0.73). El problema son las PCs de gradiente.

Idea: simular solo las 13 PCs de presión (lineal, funciona) y **calcular el gradiente** del campo de presión reconstruido → proyectarlo en las EOFs del gradiente. La no linealidad del gradiente (cuadrado de la derivada) genera la asimetría y las colas de forma física.

1. PCs de gradiente reconstruidas para 1985–2025 (6-horario) vs reales.
2. XGB: (a) 30 PCs reales, (b) msl + gradiente reconstruido, (c) solo msl.
3. Cadena: PVAR solo msl → gradiente reconstruido → XGB (b) → máx. anual de f vs histórico.

**Ejecutar en el clúster** (necesita `bluemath_tk` y los modelos PCA de 01).

In [1]:
import os, pickle, json, time
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from bluemath_tk.datamining import PCA
from bluemath_tk.core.operations import spatial_gradient
from utils.tv_var.harmonic_standardizer import HarmonicStandardizer
from utils.tv_var.periodic_var import PeriodicVAR

In [2]:
#### Modelos PCA de 01 (EOFs, escalado, coordenadas)
def cargar_pca(f):
    try:
        return PCA.load_model(f)
    except Exception:
        return pickle.load(open(f, "rb"))

pca_msl = cargar_pca("outputs/pca/pca_msl.pkl")
pca_grad = cargar_pca("outputs/pca/pca_msl_gradient.pkl")
TIME = pca_msl.pca_dim_for_rows
VAR_M = pca_msl.vars_to_stack[0]
VAR_G = pca_grad.vars_to_stack[0]

pcs_sel = json.load(open("outputs/seleccion_6h/seleccion.json"))["pcs"]
cols_msl = [c_ for c_ in pcs_sel if c_.startswith("msl")]
cols_grad = [c_ for c_ in pcs_sel if c_.startswith("grad")]
pcs_msl_h = pd.read_csv("outputs/pca/pcs_msl.csv", index_col=0, parse_dates=True)        # 13 PCs de presión
pcs_grad_h = pd.read_csv("outputs/pca/pcs_msl_gradient.csv", index_col=0, parse_dates=True)
seis = lambda df: df[df.index.hour.isin([0, 6, 12, 18])]
pcs_msl_6h, pcs_grad_6h = seis(pcs_msl_h), seis(pcs_grad_h)

In [3]:
def grad_desde_msl(pcs_m):
    """PCs de presión (todas, 13) → campo de presión → gradiente → PCs de gradiente (las 50). Por años, para no llenar la memoria."""
    trozos = []
    for ano, p in pcs_m.groupby(pcs_m.index.year):
        da = xr.DataArray(p.values, dims=(TIME, "n_component"),
                          coords={TIME: p.index.values, "n_component": np.arange(p.shape[1])})
        rec = pca_msl.inverse_transform(da)[VAR_M]
        g = spatial_gradient(rec.rename({TIME: "time"}).transpose("time", "latitude", "longitude")).rename({"time": TIME})
        pg = pca_grad.transform(g.to_dataset(name=VAR_G))["PCs"]
        trozos.append(pd.DataFrame(pg.values, index=p.index, columns=[f"grad_PC{i + 1}" for i in range(pg.shape[1])]))
    return pd.concat(trozos)

t = time.time()
pcs_grad_rec = grad_desde_msl(pcs_msl_6h.loc["1985":"2025"])
pcs_grad_rec.to_csv("outputs/pca/pcs_grad_reconstruido_6h_1985_2025.csv")
print(f"{len(pcs_grad_rec)} pasos en {time.time() - t:.0f} s")

Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_d

59900 pasos en 27 s


In [4]:
#### 1. Gradiente reconstruido vs real: correlación, std y colas por PC seleccionada
from scipy.stats import skew
real = pcs_grad_6h.reindex(pcs_grad_rec.index)
tabla1 = pd.DataFrame({"corr": [real[c_].corr(pcs_grad_rec[c_]) for c_ in cols_grad],
                       "std rec/real": [pcs_grad_rec[c_].std() / real[c_].std() for c_ in cols_grad],
                       "asim real": [skew(real[c_]) for c_ in cols_grad],
                       "asim rec": [skew(pcs_grad_rec[c_]) for c_ in cols_grad],
                       "q999 rec/real": [pcs_grad_rec[c_].quantile(0.999) / real[c_].quantile(0.999) for c_ in cols_grad]},
                      index=cols_grad)
print(tabla1.round(2).to_string())

           corr  std rec/real  asim real  asim rec  q999 rec/real
grad_PC2   0.89          0.75       0.46      0.34           0.64
grad_PC3   0.87          0.82       0.42      0.58           0.78
grad_PC4   0.78          0.67       0.51      0.01           0.57
grad_PC5   0.76          0.65       0.49      0.13           0.61
grad_PC9   0.62          0.60       0.06     -0.76           0.63
grad_PC10  0.64          0.57       0.05      0.13           0.66
grad_PC11  0.59          0.61       0.02     -0.14           0.66
grad_PC12  0.61          0.61      -0.25     -0.20           0.67
grad_PC14  0.60          0.65       0.29      0.41           0.65
grad_PC16  0.46          0.62      -0.04      0.23           0.65
grad_PC21  0.44          0.48       0.09     -0.13           0.37
grad_PC23  0.52          0.57      -0.02     -0.10           0.61
grad_PC33  0.39          0.47      -0.09      0.41           0.57
grad_PC35  0.45          0.50      -0.13     -0.42           0.53
grad_PC43 

In [5]:
#### 2. XGB con las mismas lags y parámetros que 04B
z_bulk = pd.read_csv("outputs/marginal/z_bulk.csv", index_col=0, parse_dates=True)
z6 = z_bulk.rolling("6h", center=True).mean()
z6 = z6[z6.index.hour.isin([0, 6, 12, 18])]["VHM0"].dropna()
LAGS = [0, 6, 12, 18, 24, 30, 36]
PAR = dict(n_estimators=300, learning_rate=0.05, max_depth=6, min_child_weight=200, subsample=0.8,
           colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=0)

def lagged(p):
    X = pd.concat({f"l{l}": p.shift(freq=pd.Timedelta(hours=l)) for l in LAGS}, axis=1).reindex(p.index)
    X.columns = [f"{c_}_l{l[1:]}" for l, c_ in X.columns]
    return X.dropna()

def ajustar(p):
    X = lagged(p); idx = X.index.intersection(z6.index); X, y = X.loc[idx], z6.loc[idx]
    tr = idx < "2016-01-01"
    m = XGBRegressor(**PAR).fit(X[tr], y[tr])
    f_te = pd.Series(m.predict(X[~tr]), index=X.index[~tr])
    d = lambda s: s.resample("D").mean()
    return m, list(X.columns), {"corr 6h test": f_te.corr(y[~tr]), "corr diaria test": d(f_te).corr(d(y[~tr]))}

pm = pcs_msl_6h.loc["1985":"2025"][cols_msl]
variantes = {"(a) 30 PCs reales": pm.join(pcs_grad_6h.loc["1985":"2025"][cols_grad]),
             "(b) msl + grad reconstruido": pm.join(pcs_grad_rec[cols_grad]),
             "(c) solo msl": pm}
modelos, filas = {}, {}
for k, p in variantes.items():
    m, cols, met = ajustar(p); modelos[k] = (m, cols); filas[k] = met
print(pd.DataFrame(filas).T.round(3).to_string())

                             corr 6h test  corr diaria test
(a) 30 PCs reales                   0.776             0.810
(b) msl + grad reconstruido         0.738             0.771
(c) solo msl                        0.726             0.759


In [6]:
#### 3. Cadena: PVAR solo con las 13 PCs de presión → gradiente reconstruido → XGB (b)
harm_m = HarmonicStandardizer(n_harmonics=3).fit(pcs_msl_6h)
var_m = PeriodicVAR(p=10, n_harm=1, freq="6h").fit(harm_m.transform(pcs_msl_6h))
print(var_m.stability() if hasattr(var_m, "stability") else "")

def am_f(m, cols, p):
    X = lagged(p)[cols]
    f = pd.Series(m.predict(X), index=X.index)
    return f.groupby(f.index.year).max().mean()

m_b, cols_b = modelos["(b) msl + grad reconstruido"]
m_c, cols_c = modelos["(c) solo msl"]
hist_b = am_f(m_b, cols_b, pm.join(pcs_grad_rec[cols_grad]))
hist_c = am_f(m_c, cols_c, pm)

res = []
for i in range(3):
    t = time.time()
    sim = harm_m.inverse_transform(var_m.simulate(n_years=41, start="1985-01-01", seed=i, burn=1460))
    g_sim = grad_desde_msl(sim)
    res.append({"(b) msl sim + grad reconstruido": am_f(m_b, cols_b, sim[cols_msl].join(g_sim[cols_grad])),
                "(c) solo msl sim": am_f(m_c, cols_c, sim[cols_msl])})
    print(f"sim {i} ({time.time() - t:.0f} s)", res[-1])
res = pd.DataFrame(res).mean()
print(f"máx. anual de f — (b): hist {hist_b:.2f} | PVAR msl + gradiente {res.iloc[0]:.2f} | ratio {res.iloc[0] / hist_b:.2f}")
print(f"máx. anual de f — (c): hist {hist_c:.2f} | PVAR msl {res.iloc[1]:.2f} | ratio {res.iloc[1] / hist_c:.2f}")
print("referencia: 30 PCs con PVAR o surrogado lineal → ratio ≈ 0.73")

{'rho': 0.2567433887256333, 'rho_daily': 0.9962817840481909, 'rho_step': 0.9990691470877157, 'stable': True}


Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_d

sim 0 (24 s) {'(b) msl sim + grad reconstruido': np.float32(2.3860316), '(c) solo msl sim': np.float32(2.482185)}


Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_d

sim 1 (23 s) {'(b) msl sim + grad reconstruido': np.float32(2.3022416), '(c) solo msl sim': np.float32(2.4072845)}


Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 1800 out of 1800 available variables 
If this is originated by using few times, please check 'nan_threshold_to_d

sim 2 (22 s) {'(b) msl sim + grad reconstruido': np.float32(2.3437016), '(c) solo msl sim': np.float32(2.4115481)}
máx. anual de f — (b): hist 2.45 | PVAR msl + gradiente 2.34 | ratio 0.96
máx. anual de f — (c): hist 2.51 | PVAR msl 2.43 | ratio 0.97
referencia: 30 PCs con PVAR o surrogado lineal → ratio ≈ 0.73


**Lectura**
- Si (b) apenas pierde correlación frente a (a) y la cadena da ratio ≈ 1 → **solución**: PVAR de 13 PCs de presión + gradiente calculado. Generador lineal, pequeño y físico; el kNN deja de ser necesario.
- Si (b) pierde mucho → el gradiente reconstruido con 13 PCs no basta (más PCs de presión, o quedarse con el kNN).
- (c) es el plan B: XGB solo con presión (corr test ≈ 0.73 frente a 0.78) con un PVAR que funciona tal cual.